# Beispiele: Modelle in der Praxis

Was ein Modell eigentlich ist und wie man damit arbeitet: eine Pipeline gegen eine Baseline trainieren, hineinschauen (Architektur, Parameter, Hyperparameter), mit Metadaten speichern, laden und prüfen, sehen, was ohne Scaler passiert, Drift simulieren, auf drei Arten neu trainieren, mit `partial_fit` weitertrainieren und zum Schluss eine Model Card schreiben.

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten werden erzeugt (Kündigungen von Kunden über zwölf Monate).
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/09_Modelle_Praxis.ipynb)
- **Zufall:** `SEED = 42`. Gespeichert wird in einen temporären Ordner, das Notebook legt also keine Dateien im Repository an.
- **Kapitel in der Lernapp:** «Modelle in der Praxis».

In [ ]:
import json
import tempfile
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
ORDNER = Path(tempfile.mkdtemp(prefix="modelle_"))
FEATURES = ["alter", "einkommen_kchf", "kunde_seit_monaten"]
print("scikit-learn", sklearn.__version__, "· Ablage:", ORDNER)

## 1 · Daten und Baseline

Ein Abo-Anbieter will vorhersagen, welche Kunden kündigen (`kuendigt = 1`). Die Funktion `daten(monat)` erzeugt Kunden zu einem bestimmten Monat. Sie enthält absichtlich **Drift**, die erst später wichtig wird:

- **Data Drift:** Die Kunden werden mit jedem Monat im Schnitt ein Jahr jünger.
- **Concept Drift:** Am Anfang kündigen eher ältere Kunden, nach einigen Monaten (z. B. wegen eines neuen Konkurrenzangebots) eher jüngere.

Trainiert wird auf Monat 0. Zuerst splitten, dann die Baseline (immer die häufigste Klasse).

In [ ]:
def daten(monat, n=2000, seed=SEED):
    rng = np.random.default_rng(seed + 1000 * monat)
    alter = rng.normal(45 - 1.0 * monat, 12, n).clip(18, 90)
    einkommen = rng.normal(85, 25, n).clip(20, 250)
    dauer = rng.exponential(36, n)
    logit = 0.08 * (alter - 45) * (1 - monat / 9) - 0.03 * (dauer - 36) + 0.02 * (einkommen - 85)
    y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return pd.DataFrame({"alter": alter, "einkommen_kchf": einkommen, "kunde_seit_monaten": dauer, "kuendigt": y})


df0 = daten(0)
X_trval, X_test0, y_trval, y_test0 = train_test_split(df0[FEATURES], df0["kuendigt"], test_size=0.2, stratify=df0["kuendigt"], random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=SEED)
print(f"Train {len(X_train)}, Validation {len(X_val)}, Test {len(X_test0)} Zeilen; Anteil Kündigungen {y_train.mean():.2f}")

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Baseline (häufigste Klasse): Validation-Accuracy {accuracy_score(y_val, baseline.predict(X_val)):.3f} (n = {len(y_val)})")

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])
pipe.fit(X_train, y_train)
print(f"Pipeline: Validation-Accuracy {accuracy_score(y_val, pipe.predict(X_val)):.3f} (n = {len(y_val)})")

## 2 · Was steckt im Modell?

Ein Modell besteht aus **Architektur** (hier: Standardisieren, dann gewichtete Summe und Sigmoid), **Parametern** (beim `fit` gelernt) und **Hyperparametern** (von uns gewählt, steuern das Lernen). Bei einer Pipeline gehören auch die gelernten Werte des Scalers dazu: Mittelwert und Standardabweichung jeder Spalte.

In [ ]:
scaler, modell = pipe.named_steps["scaler"], pipe.named_steps["model"]
parameter = pd.DataFrame({"scaler_mean": scaler.mean_, "scaler_std": scaler.scale_, "gewicht": modell.coef_[0]}, index=FEATURES)
print("Architektur:", pipe)
print("\nGelernte Parameter:")
print(parameter.round(3))
print("Bias (intercept):", round(float(modell.intercept_[0]), 3))
n_param = scaler.mean_.size + scaler.scale_.size + modell.coef_.size + modell.intercept_.size
print("Anzahl gelernter Zahlen:", n_param)
print("\nEinige Hyperparameter:", {k: modell.get_params()[k] for k in ["C", "max_iter", "solver", "tol"]})

## 3 · Speichern mit Metadaten

Gespeichert wird die **ganze Pipeline** mit `joblib`, daneben eine JSON-Datei mit allem, was man später braucht, um das Modell zu verstehen und neu zu trainieren.

In [ ]:
def speichern(pipe, version, X_tr, y_tr, acc, base_acc, gemessen_auf, extra=None):
    datei = ORDNER / f"modell_{version}.joblib"
    joblib.dump(pipe, datei)
    meta = {
        "version": version,
        "datum": str(date.today()),
        "sklearn": sklearn.__version__,
        "features": list(X_tr.columns),
        "seed": SEED,
        "n_train": int(len(X_tr)),
        "accuracy": round(float(acc), 3),
        "gemessen_auf": gemessen_auf,
        "baseline_accuracy": round(float(base_acc), 3),
        **(extra or {}),
    }
    (ORDNER / f"modell_{version}.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return datei, meta


val_acc_v1 = accuracy_score(y_val, pipe.predict(X_val))
base_acc_v1 = accuracy_score(y_val, baseline.predict(X_val))
datei_v1, meta_v1 = speichern(pipe, "v1", X_train, y_train, val_acc_v1, base_acc_v1, f"Validation Monat 0, n = {len(y_val)}", {"trainiert_auf": "Monat 0"})
print(f"{datei_v1.name}: {datei_v1.stat().st_size / 1024:.1f} KB")
print(json.dumps(meta_v1, indent=2))

## 4 · Laden und prüfen

Nach dem Laden muss das Modell auf denselben Daten genau dasselbe vorhersagen. Diese eine Zeile Test lohnt sich immer. (`joblib.load` nur für eigene oder vertrauenswürdige Dateien verwenden: Beim Laden wird Code ausgeführt.)

In [ ]:
geladen = joblib.load(datei_v1)
assert (geladen.predict(X_val) == pipe.predict(X_val)).all()
print("Geladenes Modell liefert dieselben Vorhersagen.")

meta = json.loads((ORDNER / "modell_v1.json").read_text(encoding="utf-8"))
assert meta["sklearn"] == sklearn.__version__, "Anderes scikit-learn: Modell neu trainieren oder Version angleichen"
assert list(X_val.columns) == meta["features"], "Spalten oder Reihenfolge stimmen nicht"

## 5 · Falle: nur das Modell ohne Scaler

Was passiert, wenn man nur den Schätzer speichert und beim Vorhersagen die Rohwerte hineingibt? Es gibt **keine Fehlermeldung**, nur schlechte Vorhersagen.

In [ ]:
nur_modell = pipe.named_steps["model"]
acc_ohne = accuracy_score(y_test0, nur_modell.predict(X_test0.to_numpy()))
acc_mit = accuracy_score(y_test0, pipe.predict(X_test0))
acc_base = accuracy_score(y_test0, baseline.predict(X_test0))
print(f"Testset Monat 0 (n = {len(y_test0)}): Pipeline {acc_mit:.3f} · nur Modell ohne Scaler {acc_ohne:.3f} · Baseline {acc_base:.3f}")

## 6 · Drift: das alte Modell im Einsatz

Das Modell `v1` bleibt unverändert, die Kunden ändern sich. Für jeden Monat erzeugen wir neue Daten und messen zwei Dinge: das mittlere Alter der Eingaben (Data Drift, sofort messbar) und die Accuracy (erst messbar, wenn man weiss, wer wirklich gekündigt hat).

In [ ]:
zeilen = []
for monat in range(0, 13, 2):
    d = daten(monat, n=1500, seed=SEED + 7)
    zeilen.append({
        "monat": monat,
        "mittleres_alter": d["alter"].mean(),
        "accuracy_v1": accuracy_score(d["kuendigt"], geladen.predict(d[FEATURES])),
        "baseline": max(d["kuendigt"].mean(), 1 - d["kuendigt"].mean()),
    })
verlauf = pd.DataFrame(zeilen)
print(verlauf.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(verlauf["monat"], verlauf["mittleres_alter"], marker="o")
ax[0].axhline(X_train["alter"].mean(), ls="--", c="grey", label="Training (Monat 0)")
ax[0].set(title="Data Drift: mittleres Alter der Eingaben", xlabel="Monat", ylabel="Jahre")
ax[0].legend()
ax[1].plot(verlauf["monat"], verlauf["accuracy_v1"], marker="o", label="Modell v1")
ax[1].plot(verlauf["monat"], verlauf["baseline"], ls="--", c="grey", label="Baseline")
ax[1].set(title="Accuracy des alten Modells", xlabel="Monat", ylim=(0.4, 1))
ax[1].legend()
plt.tight_layout()
plt.show()

## 7 · Neu trainieren: drei Varianten

Im Monat 12 liegen neue, gelabelte Daten vor. Wir vergleichen drei Varianten mit **derselben Pipeline**, auf **zwei festen Testsets**: dem alten Testset aus Monat 0 (wurde etwas vergessen?) und einem neuen Testset aus Monat 12 (ist die Drift aufgefangen?). Die neuen Testdaten werden vor dem Training abgetrennt und nie zum Trainieren benutzt.

In [ ]:
df12 = daten(12)
X12_tr, X12_test, y12_tr, y12_test = train_test_split(df12[FEATURES], df12["kuendigt"], test_size=0.2, stratify=df12["kuendigt"], random_state=SEED)


def neue_pipeline():
    return Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression())])


varianten = {
    "A · altes Modell v1 behalten": geladen,
    "B · neu, nur Monat 12": neue_pipeline().fit(X12_tr, y12_tr),
    "C · neu, Monat 0 + Monat 12": neue_pipeline().fit(pd.concat([X_train, X12_tr]), pd.concat([y_train, y12_tr])),
}
base_alt = max(y_test0.mean(), 1 - y_test0.mean())
base_neu = max(y12_test.mean(), 1 - y12_test.mean())
vergleich = pd.DataFrame(
    {name: {"Test Monat 0": accuracy_score(y_test0, m.predict(X_test0)), "Test Monat 12": accuracy_score(y12_test, m.predict(X12_test))} for name, m in varianten.items()}
).T
vergleich.loc["Baseline (häufigste Klasse)"] = [base_alt, base_neu]
print(f"n: Test Monat 0 = {len(y_test0)}, Test Monat 12 = {len(y12_test)}")
print(vergleich.round(3))

Lesart: Das alte Modell ist auf den aktuellen Daten sogar schlechter als die Baseline, obwohl sich an ihm nichts geändert hat. Variante B ist für heute am besten, hat aber das Verhalten von Monat 0 verlernt; das ist hier gewollt, weil der alte Zusammenhang nicht mehr gilt. Variante C mittelt zwei widersprüchliche Zusammenhänge und ist auf beiden Testsets nur mittelmässig. Bei reinem Data Drift ohne Concept Drift wäre C oft die beste Wahl.

## 8 · Weitertrainieren statt neu: `partial_fit`

Grosse Modelle trainiert man selten von null neu, sondern lernt vom gespeicherten Stand aus weiter. `SGDClassifier` kann das mit `partial_fit`. Wir trainieren zuerst einige Epochs auf Monat 0, dann **nur noch auf Monat 12** und beobachten beide Testsets: Das ist **Catastrophic Forgetting** in klein. Der Scaler bleibt der von Monat 0, wie bei einem eingefrorenen Teil eines Netzes.

In [ ]:
sc = StandardScaler().fit(X_train)
sgd = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, random_state=SEED)
klassen = np.array([0, 1])
protokoll = []
for epoch in range(1, 21):
    daten_epoch = (X_train, y_train) if epoch <= 10 else (X12_tr, y12_tr)
    sgd.partial_fit(sc.transform(daten_epoch[0]), daten_epoch[1], classes=klassen)
    protokoll.append({
        "epoch": epoch,
        "trainiert_auf": "Monat 0" if epoch <= 10 else "Monat 12",
        "acc_test_monat_0": accuracy_score(y_test0, sgd.predict(sc.transform(X_test0))),
        "acc_test_monat_12": accuracy_score(y12_test, sgd.predict(sc.transform(X12_test))),
    })
protokoll = pd.DataFrame(protokoll)
print(protokoll.iloc[[0, 4, 9, 10, 14, 19]].round(3).to_string(index=False))

plt.figure(figsize=(6, 3.5))
plt.plot(protokoll["epoch"], protokoll["acc_test_monat_0"], marker="o", label="Test Monat 0")
plt.plot(protokoll["epoch"], protokoll["acc_test_monat_12"], marker="o", label="Test Monat 12")
plt.axvline(10.5, c="grey", ls="--")
plt.text(10.7, 0.45, "ab hier nur Monat 12", fontsize=9)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim(0.4, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 9 · Neue Version speichern und eine Model Card schreiben

Variante B wird `v2`. Sie ersetzt `v1` nur, weil sie auf dem aktuellen Testset klar besser ist. `v1` bleibt im Ordner liegen (Rollback). Die Model Card beschreibt in wenigen Zeilen, was ein anderer Mensch wissen muss.

In [ ]:
v2 = varianten["B · neu, nur Monat 12"]
datei_v2, meta_v2 = speichern(
    v2, "v2", X12_tr, y12_tr,
    acc=accuracy_score(y12_test, v2.predict(X12_test)), base_acc=base_neu,
    gemessen_auf=f"Test Monat 12, n = {len(y12_test)}",
    extra={"trainiert_auf": "Monat 12", "ersetzt": "v1", "grund": "Concept Drift: Accuracy v1 auf Monat 12 unter der Baseline"},
)

model_card = f"""# Model Card: Kündigungsmodell {meta_v2['version']}

- **Zweck:** Wahrscheinlichkeit, dass ein Kunde kündigt, um ihm vorher ein Angebot zu machen. Nicht gedacht für Preis- oder Kreditentscheide.
- **Modell:** Pipeline aus StandardScaler und logistischer Regression (scikit-learn {meta_v2['sklearn']}), Seed {SEED}.
- **Daten:** {meta_v2['n_train']} Kunden aus {meta_v2['trainiert_auf']}, Features: {', '.join(meta_v2['features'])}.
- **Metrik:** Accuracy {meta_v2['accuracy']} ({meta_v2['gemessen_auf']}), Baseline {meta_v2['baseline_accuracy']}.
- **Grenzen:** Erzeugte Daten. Der Zusammenhang hat sich schon einmal gedreht (Concept Drift); monatlich Alter der Eingaben und Accuracy prüfen.
- **Versionen:** ersetzt v1 (Monat 0). v1 bleibt gespeichert für einen Rollback.
"""
(ORDNER / "MODEL_CARD_v2.md").write_text(model_card, encoding="utf-8")
print(model_card)
print("Dateien:", sorted(p.name for p in ORDNER.iterdir()))

## Zusammenfassung

- Ein Modell ist **Architektur + gelernte Parameter**; die Hyperparameter haben das Lernen gesteuert. Zur Vorhersage gehört die Vorverarbeitung dazu, darum immer die **ganze Pipeline** speichern.
- Neben die Modelldatei gehören **Metadaten** (Version, Datum, Library-Versionen, Features, Seed, Metriken mit Baseline) und das Laden wird einmal geprüft.
- Jedes Modell veraltet: **Data Drift** sieht man sofort an den Eingaben, **Concept Drift** erst an der Metrik.
- Neu trainieren mit derselben Pipeline, neue und alte Version auf **festen Testsets** vergleichen, alte Version für einen **Rollback** behalten.
- Wer nur auf neuen Daten weitertrainiert, riskiert **Catastrophic Forgetting**; ob das schlimm ist, hängt davon ab, ob die alten Daten noch gelten.